# 单例模式：几种实现方式

单例模式（Singleton）的目标：**一个类在整个进程中只有一个实例，任何地方获取到的都是同一个对象**。典型场景：全局配置、日志对象、数据库连接池、缓存管理器等。

Python 实现单例不止一条路，本篇一次看全 5 种常见写法，最后补充线程安全版本与选型建议：

1. 模块级单例 —— 最 Pythonic，日常首选
2. 装饰器实现 —— 一个装饰器复用到任意类
3. `__new__` 实现 —— 在对象创建环节拦截
4. 元类实现 —— 在"调用类"这一环节拦截
5. Borg 共享状态 —— 对象不唯一，但状态唯一

> 相关笔记：`3-__new__和__init__：继承与单例.ipynb` 详细讲过 `__new__` 单例；`2-metaclass: 元类创建类并用于单例.ipynb` 详细讲过元类单例。本篇对这两种只做简要回顾，重点补全其余方式。

## 先看全局：五种方式速览

| 方式 | 核心机制 | `a is b` | 线程安全 | 点评 |
| --- | --- | --- | --- | --- |
| 模块级 | `sys.modules` 导入缓存 | True | 是（导入自带锁） | 最简单直接，首选 |
| 装饰器 | 闭包字典缓存 | True | 否 | 通用，但装饰后"类"变成函数 |
| `__new__` | 类属性缓存 | True | 否 | 常用；注意 `__init__` 重复执行 |
| 元类 | 重写元类 `__call__` | True | 否 | 优雅、可复用给任意类 |
| Borg | 所有实例共享 `__dict__` | False | 否 | 严格说是"共享状态"，不是单例 |

## 1. 方式一：模块级单例（最 Pythonic）

Python 的模块本身就是天然单例：模块代码只在**第一次被 import 时执行一次**，之后该模块被缓存在 `sys.modules` 中，后续所有 `import` 拿到的都是同一个模块对象，模块里的全局变量自然也只有一份。

先"造"一个模块文件（notebook 中用 `%%writefile` 写出 `app_config.py`）：

In [1]:
%%writefile app_config.py
"""模块级单例：模块在进程中只会被导入一次。"""


class AppConfig:
    def __init__(self):
        self.debug = True
        self.language = "zh-CN"


# 首次导入时创建唯一实例，其他代码通过 import 共享它
settings = AppConfig()

Writing app_config.py


In [2]:
import sys

import app_config
from app_config import settings

# 两种导入姿势拿到的是同一个对象
print(app_config.settings is settings)       # True

# 模块被缓存：sys.modules 里能找到它
print("app_config" in sys.modules)           # True

# 注意：模块单例 ≠ 类单例，直接再实例化仍会得到新对象
print(app_config.AppConfig() is settings)    # False

True
True
False


**优点**：代码最少、线程安全（导入机制自带锁）、IDE 类型提示友好。

**局限**：实例在 import 时立即创建（无法延迟初始化），也不方便携带初始化参数。需要延迟加载时，可在模块中提供一个访问函数，第一次调用时才创建。

## 2. 方式二：装饰器实现

用一个装饰器把"缓存实例"的逻辑包在类外面：装饰后 `Logger` 这个名字指向一个闭包函数，每次写 `Logger()` 实际是在调用 `get_instance()`，由它保证只创建一次。好处是一个装饰器可以复用到任意类。

In [3]:
import functools


def singleton(cls):
    """装饰器版单例：用闭包字典缓存每个类的唯一实例。"""
    instances = {}

    @functools.wraps(cls)
    def get_instance(*args, **kwargs):
        if cls not in instances:
            instances[cls] = cls(*args, **kwargs)
        return instances[cls]

    return get_instance


@singleton
class Logger:
    def __init__(self):
        self.logs = []


log1 = Logger()
log2 = Logger()
print(log1 is log2)              # True
print(log1.__class__.__name__)   # Logger（实例仍是 Logger 类型）
print(type(Logger).__name__)     # function（但 Logger 这个名字本身已是函数）

True
Logger
function


In [4]:
# 类身份丢失带来的副作用：isinstance 和继承都会失效
try:
    isinstance(log1, Logger)
except TypeError as e:
    print("isinstance 失败:", e)

try:
    class TextLogger(Logger):   # Logger 已经是函数，无法被继承
        pass
except TypeError as e:
    print("继承失败:", type(e).__name__)

isinstance 失败: isinstance() arg 2 must be a type, a tuple of types, or a union
继承失败: TypeError


**注意**：装饰器方案牺牲了"类身份"——`Logger` 变成了函数，`isinstance` 会报错，也无法再被继承。需要保留类身份时，改用 `__new__` 或元类。

## 3. 方式三：`__new__` 实现（简要回顾）

`__new__()` 负责创建并返回实例，在这里检查类属性 `_instance` 即可拦截后续实例化。注意 `__init__()` 在每次调用类时仍会执行，所以要用 `_initialized` 标记防止初始化逻辑重复跑。细节见 `3-__new__和__init__：继承与单例.ipynb`。

In [5]:
class Config:
    _instance = None
    _initialized = False

    def __new__(cls):
        if cls._instance is None:
            cls._instance = super().__new__(cls)
        return cls._instance

    def __init__(self):
        if self._initialized:        # 第二次进入时实例属性已是 True，直接跳过
            return
        self.environment = "production"
        Config._initialized = True


c1 = Config()
c2 = Config()
print(c1 is c2)          # True
print(c2.environment)    # production（__init__ 只完整执行了一次）

True
production


## 4. 方式四：元类实现（简要回顾）

写 `Database()` 时，实际触发的是**元类的 `__call__`**。在元类里拦截这次调用，就能在"实例化"这一层做缓存。一个元类可以给任意多个类复用，且 `_instances` 按类做 key，不同类各自拥有自己的单例。细节见 `2-metaclass: 元类创建类并用于单例.ipynb`。

In [6]:
class SingletonMeta(type):
    _instances = {}

    def __call__(cls, *args, **kwargs):
        if cls not in cls._instances:
            cls._instances[cls] = super().__call__(*args, **kwargs)
        return cls._instances[cls]


class Database(metaclass=SingletonMeta):
    pass


class Cache(metaclass=SingletonMeta):
    pass


db1, db2 = Database(), Database()
cache1, cache2 = Cache(), Cache()

print(db1 is db2)         # True
print(cache1 is cache2)   # True
print(db1 is cache1)      # False：按类做 key，不同类互不影响

True
True
False


## 5. 方式五：Borg / 共享状态（Monostate）

Borg 换了个思路：**不去限制实例的个数，而是让所有实例共享同一个属性字典**。于是 `a is b` 为 `False`（对象确实不止一个），但状态完全互通，达到"全局唯一状态"的实际效果。

In [7]:
class Borg:
    _shared_state = {}                     # 所有实例共享的属性字典

    def __init__(self):
        self.__dict__ = self._shared_state  # 关键：把自己 __dict__ 指向共享字典


a = Borg()
b = Borg()
a.theme = "dark"
b.language = "zh"

print(a is b)                    # False：对象本身可以有多个
print(a.theme, b.language)       # dark zh：a 写的属性 b 能看到
print(a.__dict__ is b.__dict__)  # True：本质是共享同一份状态

False
dark zh
True


**适用场景**：需要"状态全局唯一"但不关心对象身份，或希望单例类能被正常继承时（子类实例同样共享状态，除非覆盖 `_shared_state`）。严格来说 Borg 不是单例模式，而是它的变体 Monostate。

## 6. 进阶：线程安全版本

前面 `__new__`、装饰器、元类写法中的"检查再创建"都是**两步操作**，不是原子的。多线程同时首次调用时，可能都看到 `_instance is None`，于是各自创建一个实例——基础写法在并发下会失效。

先复现这个竞态：用 `time.sleep()` 模拟"创建耗时"，放大竞态窗口（`sleep` 会释放 GIL，其他线程得以同时进入检查）。

In [8]:
import threading
import time


class RacySingleton:
    _instance = None

    def __new__(cls):
        if cls._instance is None:
            time.sleep(0.05)                  # 放大竞态窗口
            cls._instance = super().__new__(cls)
        return cls._instance


holders = []

def create():
    holders.append(RacySingleton())


threads = [threading.Thread(target=create) for _ in range(32)]
for t in threads:
    t.start()
for t in threads:
    t.join()

unique_ids = {id(obj) for obj in holders}
print(f"32 个线程共创建了 {len(unique_ids)} 个不同实例")

32 个线程共创建了 32 个不同实例


修复方式：加锁，并用**双重检查锁定**（double-checked locking）——第一次检查放在锁外，实例已存在时避免加锁开销；第二次检查放在锁内，防止两个线程先后通过第一次检查后重复创建。

In [9]:
import threading


class SafeSingleton:
    _instance = None
    _lock = threading.Lock()

    def __new__(cls):
        if cls._instance is None:            # 第一次检查：已有实例时无需加锁
            with cls._lock:                  # 同一时刻只放一个线程进来
                if cls._instance is None:    # 第二次检查：可能刚被别的线程创建
                    cls._instance = super().__new__(cls)
        return cls._instance


holders = []

def create():
    holders.append(SafeSingleton())


threads = [threading.Thread(target=create) for _ in range(32)]
for t in threads:
    t.start()
for t in threads:
    t.join()

unique_ids = {id(obj) for obj in holders}
print(f"32 个线程共创建了 {len(unique_ids)} 个不同实例")   # 1

32 个线程共创建了 1 个不同实例


## 7. 对比与选型

| 方式 | 唯一性保证 | 线程安全 | 继承友好 | 适用场景 |
| --- | --- | --- | --- | --- |
| 模块级 | 导入缓存 | 是 | —（无类可继承） | 绝大多数"全局唯一对象"需求，首选 |
| 装饰器 | 闭包字典 | 否 | 差（丢失类身份） | 快速给现成类加单例，不涉及 isinstance/继承 |
| `__new__` | 类属性 | 否（可加锁） | 可控 | 需要保留类身份、逻辑集中在一个类里 |
| 元类 | 元类 `__call__` | 否（可加锁） | 好（可复用、按类缓存） | 多个类共用同一套单例逻辑、框架代码 |
| Borg | 共享 `__dict__` | 否 | 好 | 要"状态唯一"而非"对象唯一" |

### 使用注意点

- **多进程失效**：单例只在单个进程内成立，`multiprocessing` 的子进程会各自复制一份。跨进程唯一要靠外部设施（Redis、数据库、文件锁）。
- **`__init__` 重复执行**：`__new__` 和装饰器写法里，每次调用类仍会执行 `__init__`，需要 `_initialized` 之类的标记挡住重复初始化。
- **`copy` / `deepcopy` 会绕过单例**：复制出的新对象不受 `_instance` 约束，必要时可定义 `__copy__` / `__deepcopy__` 返回 `self`。
- **全局状态的代价**：单例本质是全局变量，单元测试难以隔离。可以提供 `reset()` 钩子方便测试，或改用依赖注入显式传对象。

## 一句话总结

Python 里实现单例本质都是"**把第一次的结果缓存起来，之后一律返回缓存**"，区别只在缓存放在哪：模块放 `sys.modules`、装饰器放闭包字典、`__new__` 放类属性、元类放 `_instances`；Borg 则不缓存对象、只共享 `__dict__`。日常优先用模块级单例，需要保留类身份或多类复用时选 `__new__` / 元类，并发场景记得加双重检查锁。